### Lab 3.2 Backpropagation

In this lab you will inspect the gradients in a neural network and understand how they are computed as they propagate from the loss backwards through the network.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

Let's make some random data: a 2D input point and a label (set to one).

In [2]:
x = torch.randn(2)
y = torch.ones(1).long()

In [3]:
x, y

(tensor([-2.3374, -0.9438]), tensor([1]))

Now let's make the parameters needed for a multi-layer perceptron with a single hidden layer of three neurons.  We will be sure to set `requires_grad=True` so that PyTorch knows to compute the gradients for these tensors.

In [4]:
W1 = torch.randn(3,2,requires_grad=True)
b1 = torch.randn(3,requires_grad=True)

W2 = torch.randn(1,3,requires_grad=True)
b2 = torch.randn(1,requires_grad=True)

Now we compute the score output and a squared error loss term.  This is the "forward" step as explained in the backpropagation notes.

In [5]:
h = W1@x+ b1
s = F.relu(h)

z = W2@s + b2

L = 0.5*(z-y)**2

`retain_grad()` tells PyTorch not to throw away the gradients of intermediate tensors.

In [6]:
h.retain_grad()
s.retain_grad()
z.retain_grad()
L.retain_grad()

Now we call `backward()` to compute the gradients.

In [7]:
L.backward()

Let's think about what the gradient of the loss w.r.t. $z$ should be.
$$L = \frac{1}{2}(z-y)^2$$
$$\frac{dL}{dz} = (z-y)$$

In [8]:
dLdz = z-y

Let's check our answer with PyTorch's answer.

In [9]:
dLdz, z.grad

(tensor([-5.0670], grad_fn=<SubBackward0>), tensor([-5.0670]))

Yup, they're the same!

Now, we work backward to obtain the gradients for $W^2$ and $\vec{b}^2$ as explained in the backprogation notes.

In [10]:
dLdW2 = dLdz[:,None] @ s[None,:]

In [11]:
dLdW2, W2.grad

(tensor([[ -4.9634, -11.4899,  -4.0049]], grad_fn=<MmBackward0>),
 tensor([[ -4.9634, -11.4899,  -4.0049]]))

In [12]:
dzdb2 = torch.eye(1)

In [13]:
dLdb2 = dLdz @ dzdb2

Note that it would have been more efficient to simply do `dLdb2 = dLdz` since multiplying by $1$ has no effect.

In [14]:
dLdb2, b2.grad

(tensor([-5.0670], grad_fn=<SqueezeBackward4>), tensor([-5.0670]))

### Exercises

Continue to work back until you can calculate the derivatives w.r.t. $W_1$ and $\vec{b_1}$.

1. Calculate $dL/d\vec{s}$ and check your answer.


In [15]:
# dL/ds = dL/dz * dz/ds
# dL/dz calculated above
# dz/ds = d/ds(W2*s + b) = W2

dLds = dLdz @ W2

In [17]:
dLds, s.grad

(tensor([7.4584, 3.8655, 4.2352], grad_fn=<SqueezeBackward4>),
 tensor([7.4584, 3.8655, 4.2352]))

2. Compute $dL/d\vec{h}$ and check your answer.

In [18]:
# dL/dh = dL/ds * ds/dh
# dL/ds calculated above
# ds/dh = d/dh(ReLU(h)) = [x > 0]

dLdh = dLds * (h > 0)

In [19]:
dLdh, h.grad

(tensor([7.4584, 3.8655, 4.2352], grad_fn=<MulBackward0>),
 tensor([7.4584, 3.8655, 4.2352]))

3. Compute $dL/dW^1$ and $dL/d\vec{b^1}$ and check your answers.

In [43]:
# dL/dW1 = dL/dh^T * x^T
dLdW1 = dLdh[:,None] @ x[None,:]
dLdW1, W1.grad

(tensor([[-17.4331,  -7.0394],
         [ -9.0352,  -3.6484],
         [ -9.8994,  -3.9973]], grad_fn=<MmBackward0>),
 tensor([[-17.4331,  -7.0394],
         [ -9.0352,  -3.6484],
         [ -9.8994,  -3.9973]]))

In [44]:
# DL/db1 = dL/dh^T * I
dLdb1 = dLdh[:,None].T @ torch.eye(3)
dLdb1, b1.grad

(tensor([[7.4584, 3.8655, 4.2352]], grad_fn=<MmBackward0>),
 tensor([7.4584, 3.8655, 4.2352]))